# RFC Simulator: Triad-First Deterministic Closure + Validation Screens

This notebook is the revised simulator for **Recursive Fractal Cosmology: A Generative Ontology of Existence**.

It preserves the current canonical derivation spine:

```text
CIF-QV-RFL triad
-> Module G: deterministic triadic closure
-> Module R: triad-grouped global closure audit
-> Module N V2: dimensional projection bridge
-> Module S: one-anchor SI bridge
-> Module T: dimensionless coupling map
```

It also adds the new downstream validation-screen family:

```text
Module U  : one-anchor constant table screen
Module V  : precision cosmology compressed-parameter screen
Module W  : BBN light-abundance proxy screen
Module X  : CP/EDM bound screen
Module Y2 : particle-sector refinement candidate screen
Module Z  : observer / neural validation harness
Module QG : finite spin-foam transition-amplitude audit
```

Core rule:

```text
Modules consume the frozen packet.
Modules do not choose or retune the frozen packet.
Y2 is exploratory candidate discovery and must be frozen/retested before independent validation claims.
```

Deprecated modules such as `G_legacy`, `N_legacy`, and `R_legacy` remain historical only.


In [ ]:
# Core imports
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, HTML
from ipywidgets import Dropdown, Button, VBox, Output, Layout

try:
    import json5
except Exception:
    json5 = None

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True


## 1. Load configuration files

The notebook looks for:

```text
SimulationConfigs.json
Module_G_R_N_S_T_FrozenPacket.json
ValidationScreens_U_V_W_X_Y2_Z_QG.json   # optional
```

in the current folder, the repo root, or `notebooks/`.

The simulator can run from either:
- module configs in `SimulationConfigs.json`, or
- validation results embedded in the revised frozen-packet JSON under `validationScreens`.


In [ ]:
def load_json_file(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        if json5 is None:
            raise
        return json5.loads(text)


def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def normalize_configs(obj):
    if obj is None:
        return []
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ["modules", "simulationConfigs", "configs", "moduleConfigs"]:
            if isinstance(obj.get(key), list):
                return obj[key]
        # If it is a dictionary keyed by module name, convert values to list.
        vals = []
        for k, v in obj.items():
            if isinstance(v, dict):
                v = dict(v)
                v.setdefault("module", k)
                vals.append(v)
        return vals
    raise ValueError("Unsupported SimulationConfigs JSON structure.")


CONFIG_CANDIDATES = [
    Path("SimulationConfigs.json"),
    Path("notebooks") / "SimulationConfigs.json",
    Path.cwd() / "SimulationConfigs.json",
    Path.cwd() / "notebooks" / "SimulationConfigs.json"
]

PACKET_CANDIDATES = [
    Path("Module_G_R_N_S_T_FrozenPacket.json"),
    Path("notebooks") / "Module_G_R_N_S_T_FrozenPacket.json",
    Path.cwd() / "Module_G_R_N_S_T_FrozenPacket.json",
    Path.cwd() / "notebooks" / "Module_G_R_N_S_T_FrozenPacket.json"
]

VALIDATION_CANDIDATES = [
    Path("ValidationScreens_U_V_W_X_Y2_Z_QG.json"),
    Path("notebooks") / "ValidationScreens_U_V_W_X_Y2_Z_QG.json",
    Path.cwd() / "ValidationScreens_U_V_W_X_Y2_Z_QG.json",
    Path.cwd() / "notebooks" / "ValidationScreens_U_V_W_X_Y2_Z_QG.json"
]

config_path = find_first_existing(CONFIG_CANDIDATES)
packet_path = find_first_existing(PACKET_CANDIDATES)
validation_path = find_first_existing(VALIDATION_CANDIDATES)

if config_path is None:
    raise FileNotFoundError("Could not find SimulationConfigs.json in repo root or notebooks/.")

all_configs = normalize_configs(load_json_file(config_path))

frozen_packet = load_json_file(packet_path) if packet_path is not None else None
external_validation_screens = load_json_file(validation_path) if validation_path is not None else None

print(f"Loaded module configs from: {config_path}")
print(f"Number of module configs loaded: {len(all_configs)}")

if packet_path is not None:
    print(f"Loaded frozen packet from: {packet_path}")
else:
    print("No frozen-packet JSON found. Using values embedded in SimulationConfigs.json.")

if validation_path is not None:
    print(f"Loaded external validation-screen file from: {validation_path}")
else:
    print("No separate validation-screen JSON found. Will use validationScreens from frozen packet/configs if present.")


## 2. Core utilities

In [ ]:
def module_key(cfg):
    return str(cfg.get("module", cfg.get("id", "UNKNOWN")))


def get_config(selected_module):
    for cfg in all_configs:
        if module_key(cfg) == selected_module:
            return cfg
    raise KeyError(f"Module {selected_module} not found in SimulationConfigs.json")


def nested_get(obj, path, default=None):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def get_active_g_packet():
    # Revised frozen-packet schema.
    if isinstance(frozen_packet, dict):
        for key in ["moduleG", "moduleG_deterministicTriadicClosure"]:
            g = frozen_packet.get(key, {})
            if isinstance(g, dict) and "packet" in g:
                return g["packet"]
        # Some files store the packet under derivationStatus or canonicalPacket.
        for key in ["packet", "canonicalPacket", "frozenPacket"]:
            if isinstance(frozen_packet.get(key), dict):
                return frozen_packet[key]

    # Fall back to active Module G inside SimulationConfigs.json.
    for cfg in all_configs:
        if cfg.get("module") == "G" and "packet" in cfg:
            return cfg["packet"]

    return {
        "delta": 4.6692,
        "cycleLength": 60,
        "alpha": 0.0256831,
        "phaseDepthK": 2,
        "nu": 0.00420784,
        "epsilon": 0.000108071,
        "lambdaNormalized": 0.489442,
        "nClosure": 18,
        "nFullCanonical": 40
    }


G_PACKET = get_active_g_packet()


def packet_value(name, default=None):
    return G_PACKET.get(name, default)


def get_validation_screens():
    screens = {}
    if isinstance(frozen_packet, dict) and isinstance(frozen_packet.get("validationScreens"), dict):
        screens.update(frozen_packet["validationScreens"])
    if isinstance(external_validation_screens, dict):
        if isinstance(external_validation_screens.get("validationScreens"), dict):
            screens.update(external_validation_screens["validationScreens"])
        else:
            screens.update(external_validation_screens)
    # Also allow SimulationConfigs.json to include U/V/W/X/Y2/Z/QG as normal configs.
    for cfg in all_configs:
        mod = module_key(cfg)
        if mod in {"U", "V", "W", "X", "Y2", "Z", "QG"}:
            screens.setdefault(mod, cfg)
    return screens


VALIDATION_SCREENS = get_validation_screens()


def find_screen(module_code):
    # Match by direct key, module field, or key suffix.
    if module_code in VALIDATION_SCREENS:
        return VALIDATION_SCREENS[module_code]
    for k, v in VALIDATION_SCREENS.items():
        if isinstance(v, dict) and v.get("module") == module_code:
            return v
        if str(k).lower().startswith(f"module{module_code.lower()}") or str(k).lower().endswith(module_code.lower()):
            return v
    return {}


def get_delta_alpha_nu_n(cfg=None):
    cfg = cfg or {}
    delta = cfg.get("delta", packet_value("delta", 4.6692))
    alpha = cfg.get("alpha", packet_value("alpha", 0.0256831))
    nu = cfg.get("nu", packet_value("nu", 0.00420784))
    n = cfg.get("nFullCanonical", packet_value("nFullCanonical", 40))
    return float(delta), float(alpha), float(nu), int(n)


def time_grid(cfg, default_end=None):
    dt = float(cfg.get("dt", 0.1))
    t_range = cfg.get("t_range", [0, default_end or packet_value("cycleLength", 60)])
    tmin, tmax = float(t_range[0]), float(t_range[1])
    if tmax <= tmin:
        tmax = tmin + (default_end or 60)
    vals = np.arange(tmin, tmax, dt)
    if len(vals) < 2:
        vals = np.linspace(tmin, tmax, 100)
    return vals, dt, tmin, tmax


def recursive_kernel(t_vals, delta=None, alpha=None, nu=None, n=None, mode="cos"):
    delta = packet_value("delta", 4.6692) if delta is None else delta
    alpha = packet_value("alpha", 0.0256831) if alpha is None else alpha
    nu = packet_value("nu", 0.00420784) if nu is None else nu
    n = packet_value("nFullCanonical", 40) if n is None else n

    y = np.zeros_like(t_vals, dtype=float)
    for j in range(1, int(n) + 1):
        weight = (delta ** (-j)) * np.exp(-alpha * j * t_vals)
        if mode == "sin":
            y += weight * np.sin(j * t_vals + nu)
        elif mode == "rebirth":
            y += weight * np.sin(j * t_vals + j ** 2)
        else:
            y += weight * np.cos(j * t_vals + nu)
    return y


def recursive_entropy(t_vals, delta=None, alpha=None, n=None):
    delta = packet_value("delta", 4.6692) if delta is None else delta
    alpha = packet_value("alpha", 0.0256831) if alpha is None else alpha
    n = packet_value("nClosure", 18) if n is None else n

    s = np.zeros_like(t_vals, dtype=float)
    for j in range(1, int(n) + 1):
        s += j * np.log(delta) * (delta ** (-j)) * np.exp(-alpha * j * t_vals)
    return s


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj


def safe_json(data):
    return json.dumps(to_jsonable(data), indent=2, ensure_ascii=False)


def display_dict(title, data):
    display(Markdown(f"### {title}"))
    display(HTML(f"<pre style='white-space: pre-wrap; font-size: 12px'>{safe_json(data)}</pre>"))


def display_table(title, rows, headers=None):
    display(Markdown(f"### {title}"))
    if not rows:
        display(Markdown("_No rows._"))
        return
    if headers is None:
        headers = list(rows[0].keys()) if isinstance(rows[0], dict) else None

    if isinstance(rows[0], dict):
        table_rows = [[row.get(h, "") for h in headers] for row in rows]
    else:
        table_rows = rows

    html = "<table style='border-collapse: collapse; font-size: 13px;'>"
    if headers:
        html += "<tr>" + "".join(f"<th style='border:1px solid #999;padding:4px;text-align:left'>{h}</th>" for h in headers) + "</tr>"
    for row in table_rows:
        html += "<tr>" + "".join(f"<td style='border:1px solid #999;padding:4px'>{v}</td>" for v in row) + "</tr>"
    html += "</table>"
    display(HTML(html))


def bar_chart(labels, values, title, ylabel="Value", log=False):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values)
    if log:
        plt.yscale("log")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


def print_claim_boundary(cfg):
    status = cfg.get("currentStatus", cfg.get("status", "unspecified"))
    role = cfg.get("canonicalRole", "unspecified")
    may_retune = cfg.get("mayRetuneFrozenPacket", None)
    display(Markdown(f"**Current status:** `{status}`"))
    display(Markdown(f"**Canonical role:** {role}"))
    if may_retune is not None:
        display(Markdown(f"**May retune frozen packet:** `{may_retune}`"))
    if cfg.get("usesReferenceValuesForMapSelection") is True:
        display(Markdown("**Map-selection warning:** This screen uses reference values for candidate-map discovery. Freeze/retest before validation claims."))
    if "claimBoundary" in cfg:
        display(Markdown(f"**Claim boundary:** {cfg['claimBoundary']}"))


display_dict("Active frozen Module G packet", G_PACKET)
display_dict("Available validation screens", sorted(list(VALIDATION_SCREENS.keys())))


## 3. Canonical module runners: G, R, N V2, S, T

In [ ]:
def run_module_G(cfg):
    display(Markdown("## Module G: Deterministic Triadic Closure"))
    print_claim_boundary(cfg)

    packet = cfg.get("packet", G_PACKET)
    display_dict("Frozen deterministic packet", packet)

    delta = packet["delta"]
    cycle = packet["cycleLength"]
    alpha_expected = math.log(delta) / cycle
    nu_expected = packet["phaseDepthK"] * delta ** (-4)
    epsilon_expected = packet["alpha"] * packet["nu"]

    checks = cfg.get("deterministicCheckValues", {
        "empiricalTargetsUsed": cfg.get("empiricalTargetsUsed", False),
        "parameterSearchPerformed": cfg.get("parameterSearchPerformed", False),
        "mcmcUsed": cfg.get("mcmcUsed", False),
        "nutsUsed": cfg.get("nutsUsed", False),
        "alpha_expected_Log_delta_over_cycle": alpha_expected,
        "alpha_packet": packet["alpha"],
        "alpha_abs_error": abs(alpha_expected - packet["alpha"]),
        "nu_expected_phaseDepthK_delta_minus4": nu_expected,
        "nu_packet": packet["nu"],
        "nu_abs_error": abs(nu_expected - packet["nu"]),
        "epsilon_expected_alpha_times_nu": epsilon_expected,
        "epsilon_packet": packet["epsilon"],
        "epsilon_abs_error": abs(epsilon_expected - packet["epsilon"])
    })
    display_dict("Deterministic closure checks", checks)

    breakdown = cfg.get("closureBreakdownAtN18", {})
    if breakdown:
        labels = list(breakdown.keys())
        values = [breakdown[k] for k in labels]
        bar_chart(labels, values, "Module G closure breakdown at n = 18")

    t_vals = np.linspace(0, packet["cycleLength"], 800)
    psi = recursive_kernel(t_vals, packet["delta"], packet["alpha"], packet["nu"], packet["nFullCanonical"])
    S = recursive_entropy(t_vals, packet["delta"], packet["alpha"], packet["nClosure"])

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, psi, label="psi(t; nu)")
    plt.plot(t_vals, S, label="S_rec(t)")
    plt.title("Frozen Module G kernel and recursive entropy")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    symbolic = cfg.get("symbolicOutputs", {})
    if symbolic:
        display_dict("Symbolic outputs", symbolic)


def run_module_R(cfg):
    display(Markdown("## Module R: Triad-Grouped Global Closure Audit"))
    print_claim_boundary(cfg)

    keys = [
        "rawRFLResidualScore", "sourceCoupledRFLResidualScore", "residualImprovement",
        "cpResidual", "tailN18", "tailN40", "bestLag", "bestLagV1",
        "bestLagCorrelation", "bestLagCorrelationV1"
    ]
    core = {k: cfg.get(k) for k in keys if k in cfg}
    display_dict("Module R core audit values", core)

    for section in ["v1Audit", "standardizedResidualAuditV2"]:
        if isinstance(cfg.get(section), dict):
            display_dict(section, cfg[section])

    triad = cfg.get("interpretiveTriadFractions_WithDarkKernelBridge", {})
    if triad:
        bar_chart(list(triad.keys()), [triad[k] for k in triad], "Module R interpretive triad fractions")

    source = cfg.get("sourcePowerFractionsV2", {})
    if source:
        bar_chart(list(source.keys()), [source[k] for k in source], "Module R source power fractions V2")


def run_module_N(cfg):
    display(Markdown("## Module N V2: Dimensional Projection Bridge"))
    print_claim_boundary(cfg)

    for section in ["projectionFactors", "symbolicConstantsStableAcrossN", "canonicalProjectedValuesStableAcrossN", "identityResiduals", "preservedIdentities"]:
        if isinstance(cfg.get(section), dict):
            display_dict(section, cfg[section])

    proj = cfg.get("canonicalProjectedValuesStableAcrossN", {})
    if proj:
        m = proj.get("symbolicMassProjected")
        E = proj.get("meanEnergyProjected")
        alpha_em = proj.get("alphaEMProjected_inverseEnergy")
        alpha_g = proj.get("alphaGProjected_massSquared")
        Lambda = proj.get("LambdaProjected_inverseEnergyCycle")
        T = proj.get("cycleTimeProjected")
        checks = {}
        if m is not None and alpha_g is not None:
            checks["alphaG_over_mSquared"] = alpha_g / (m ** 2)
        if E is not None and alpha_em is not None:
            checks["alphaEM_times_E"] = alpha_em * E
        if E is not None and Lambda is not None and T is not None:
            checks["Lambda_times_E_times_T"] = Lambda * E * T
        display_dict("Identity checks recomputed in notebook", checks)

        labels = ["m projected", "E projected", "alpha_G projected", "Lambda projected"]
        values = [
            proj.get("symbolicMassProjected", 0),
            proj.get("meanEnergyProjected", 0),
            proj.get("alphaGProjected_massSquared", 0),
            proj.get("LambdaProjected_inverseEnergyCycle", 0)
        ]
        bar_chart(labels, values, "Module N V2 projected internal quantities")


def run_module_S(cfg):
    display(Markdown("## Module S: One-Anchor SI / Laboratory Bridge"))
    print_claim_boundary(cfg)
    display_dict("Unit bridge", cfg.get("unitBridge", {}))
    display_dict("Anchor check", cfg.get("anchorCheck", {}))

    bridge = cfg.get("unitBridge", {})
    labels = []
    values = []
    for k in ["RFC_energy_unit_MeV", "RFC_energy_unit_J", "RFC_time_unit_s", "RFC_length_unit_m", "projectedCycleTimeSI_s", "projectedCycleLengthSI_m"]:
        if k in bridge:
            labels.append(k)
            values.append(bridge[k])
    if labels:
        bar_chart(labels, values, "Module S SI bridge scales, log axis", log=True)


def run_module_T(cfg):
    display(Markdown("## Module T: Dimensionless Coupling Map"))
    print_claim_boundary(cfg)
    display_dict("Canonical map", cfg.get("canonicalMap", {}))
    display_dict("Output", cfg.get("output", {}))

    cmap = cfg.get("canonicalMap", {})
    out = cfg.get("output", {})
    raw = cmap.get("alpha_N_raw_inverse", cfg.get("rawInverseEnergyCoupling", {}).get("alpha_N_raw_inverse"))
    factor = cmap.get("screenFactor_F_T")
    mapped = out.get("alpha_T_inverse")
    ref = out.get("reference_alpha_inverse")

    if raw is not None and factor is not None:
        recomputed = raw * factor
        display_dict("Notebook recomputation", {
            "raw_inverse_energy_coupling": raw,
            "screen_factor": factor,
            "recomputed_alpha_T_inverse": recomputed,
            "stored_alpha_T_inverse": mapped,
            "reference_alpha_inverse_after_the_fact": ref
        })

    labels = [x for x in ["raw inverse", "mapped inverse", "reference inverse"]]
    values = [raw or 0, mapped or 0, ref or 0]
    bar_chart(labels, values, "Module T raw vs mapped vs reference inverse coupling")


## 4. Validation-screen runners: U, V, W, X, Y2, Z, QG

In [ ]:
def run_module_U(cfg):
    display(Markdown("## Module U: One-Anchor Constant Table Screen"))
    print_claim_boundary(cfg)

    summary = cfg.get("summary", cfg.get("audit", {}))
    selected = cfg.get("selectedResults", cfg.get("results", {}))
    display_dict("Module U summary", summary)

    rows = []
    for name, item in selected.items():
        if isinstance(item, dict):
            rows.append({
                "constant": name,
                "RFC": item.get("RFC", item.get("RFC_MeV", item.get("value"))),
                "reference": item.get("reference", item.get("reference_MeV")),
                "absPercentError": item.get("absPercentError")
            })
    display_table("Selected constant results", rows, ["constant", "RFC", "reference", "absPercentError"])

    if summary:
        labels = ["mean", "median", "max"]
        values = [
            summary.get("meanAbsPercentError", 0),
            summary.get("medianAbsPercentError", 0),
            summary.get("maxAbsPercentError", 0)
        ]
        bar_chart(labels, values, "Module U error summary", ylabel="Abs percent error")


def run_module_V(cfg):
    display(Markdown("## Module V: Precision Cosmology Compressed-Parameter Screen"))
    print_claim_boundary(cfg)
    summary = cfg.get("summary", cfg.get("results", {}))
    display_dict("Module V summary", summary)

    rows = []
    mapping = [
        ("H0", "H0ProxyErrorPercent"),
        ("Omega_m", "OmegaMProxyErrorPercent"),
        ("Omega_Lambda", "OmegaLambdaProxyErrorPercent"),
        ("sigma8", "sigma8ProxyErrorPercent"),
        ("n_s", "nsProxyErrorPercent")
    ]
    for label, key in mapping:
        if key in summary:
            rows.append({"item": label, "errorPercent": summary[key]})
    if "rProxyStatus" in summary:
        rows.append({"item": "r upper-bound proxy", "errorPercent": summary.get("rProxyStatus")})
    display_table("Compressed cosmology comparison", rows, ["item", "errorPercent"])

    numeric_rows = [r for r in rows if isinstance(r["errorPercent"], (int, float))]
    if numeric_rows:
        bar_chart([r["item"] for r in numeric_rows], [r["errorPercent"] for r in numeric_rows], "Module V compressed-parameter errors", ylabel="Percent error")

    if cfg.get("notCompleted"):
        display_dict("Not completed", cfg["notCompleted"])


def run_module_W(cfg):
    display(Markdown("## Module W: BBN Light-Abundance Proxy Screen"))
    print_claim_boundary(cfg)
    results = cfg.get("results", {})
    rows = []
    for name, item in results.items():
        if isinstance(item, dict):
            rows.append({
                "abundance": name,
                "RFC": item.get("RFC"),
                "reference": item.get("reference"),
                "absPercentError": item.get("absPercentError"),
                "status": item.get("status", "")
            })
    display_table("BBN abundance proxy results", rows, ["abundance", "RFC", "reference", "absPercentError", "status"])
    numeric = [r for r in rows if isinstance(r["absPercentError"], (int, float))]
    if numeric:
        bar_chart([r["abundance"] for r in numeric], [r["absPercentError"] for r in numeric], "Module W abundance errors", ylabel="Percent error", log=True)


def run_module_X(cfg):
    display(Markdown("## Module X: CP/EDM Bound Screen"))
    print_claim_boundary(cfg)
    results = cfg.get("results", {})
    rows = []
    for name, item in results.items():
        if isinstance(item, dict):
            rows.append({
                "item": name,
                "RFC": item.get("RFC"),
                "reference_or_bound": item.get("reference", item.get("bound")),
                "absPercentError": item.get("absPercentError", ""),
                "status": item.get("status", "")
            })
        else:
            rows.append({"item": name, "RFC": item, "reference_or_bound": "", "absPercentError": "", "status": ""})
    display_table("CP/EDM and phase-proxy results", rows, ["item", "RFC", "reference_or_bound", "absPercentError", "status"])


def run_module_Y2(cfg):
    display(Markdown("## Module Y2: Particle-Sector Refinement Screen"))
    print_claim_boundary(cfg)

    q = cfg.get("quarkMassScreen", {})
    ckm = cfg.get("CKMScreen", {})
    pmns = cfg.get("PMNSScreen", {})
    summary = cfg.get("summary", {})

    display_dict("Y2 summary", summary)

    quarks = q.get("quarks", {})
    rows = []
    for name, item in quarks.items():
        rows.append({
            "quark": name,
            "RFC_MeV": item.get("RFC_MeV"),
            "reference_MeV": item.get("reference_MeV"),
            "absPercentError": item.get("absPercentError")
        })
    display_table("QCD/log-space quark mass screen", rows, ["quark", "RFC_MeV", "reference_MeV", "absPercentError"])
    if rows:
        bar_chart([r["quark"] for r in rows], [r["absPercentError"] for r in rows], "Module Y2 quark mass errors", ylabel="Percent error")

    ckm_rows = [{"item": k, "value": v} for k, v in ckm.items()]
    pmns_rows = [{"item": k, "value": v} for k, v in pmns.items()]
    display_table("CKM screen", ckm_rows, ["item", "value"])
    display_table("PMNS screen", pmns_rows, ["item", "value"])

    if summary:
        labels = []
        values = []
        for k in ["quarkMassMeanErrorPercent", "quarkMassMaxErrorPercent", "CKMMeanAngleErrorPercent", "PMNSMeanAngleErrorPercent"]:
            if k in summary:
                labels.append(k.replace("Percent", "%"))
                values.append(summary[k])
        if labels:
            bar_chart(labels, values, "Module Y2 summary errors", ylabel="Percent error")


def run_module_Z(cfg):
    display(Markdown("## Module Z: Observer / Branching / Neural Validation Harness"))
    print_claim_boundary(cfg)

    obs = cfg.get("observerBranchingResults", {})
    neural = cfg.get("neuralEEGTargets", {})

    rows = []
    for name, item in obs.items():
        if isinstance(item, dict):
            rows.append({"item": name, "RFC": item.get("RFC"), "target": item.get("target"), "status": item.get("status")})
    display_table("Observer / branching model screen", rows, ["item", "RFC", "target", "status"])

    nrows = []
    for name, item in neural.items():
        if isinstance(item, dict):
            nrows.append({"item": name, "RFC": item.get("RFC"), "target": item.get("target"), "absPercentError": item.get("absPercentError"), "status": item.get("status")})
    display_table("Neural / EEG target signatures", nrows, ["item", "RFC", "target", "absPercentError", "status"])


def run_module_QG(cfg):
    display(Markdown("## Module QG: Finite Spin-Foam Transition-Amplitude Audit"))
    print_claim_boundary(cfg)

    results = cfg.get("results", {})
    rows = [{"item": k, "value": v} for k, v in results.items()]
    display_table("Finite spin-foam audit results", rows, ["item", "value"])

    numeric_keys = ["relativeTail", "unitarityProxy", "refinementProxy", "geometryCoherence"]
    labels, values = [], []
    for k in numeric_keys:
        if isinstance(results.get(k), (int, float)):
            labels.append(k)
            values.append(results[k])
    if labels:
        bar_chart(labels, values, "Module QG finite-audit values")


## 5. Downstream/legacy module runners

The older module family A-F and H-Q is still useful as downstream projection structure. These runners are intentionally interpreted as **projection/proxy runners**, not as packet-generation methods.


In [ ]:
def run_downstream_kernel_module(cfg):
    mod = module_key(cfg)
    display(Markdown(f"## Module {mod}: {cfg.get('description', 'Downstream projection module')}"))
    print_claim_boundary(cfg)

    if str(mod).endswith("_legacy") or cfg.get("currentStatus", "").startswith("deprecated"):
        display(Markdown("**Warning:** Historical/deprecated module. It must not be used to choose or retune the frozen packet."))
        display_dict("Legacy configuration", cfg)
        return

    t_vals, dt, _, _ = time_grid(cfg, default_end=60)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    psi = recursive_kernel(t_vals, delta, alpha, nu, n)
    S = recursive_entropy(t_vals, delta, alpha, cfg.get("nClosure", packet_value("nClosure", 18)))

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, psi, label="recursive kernel")
    plt.plot(t_vals, S, label="recursive entropy", alpha=0.8)
    plt.title(f"Module {mod}: frozen-packet downstream projection")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    metrics = {
        "kernel_min": float(np.min(psi)),
        "kernel_max": float(np.max(psi)),
        "kernel_rms": float(np.sqrt(np.mean(psi ** 2))),
        "entropy_initial": float(S[0]),
        "entropy_final": float(S[-1])
    }
    display_dict("Projection metrics", metrics)

    if "expectedDownstreamScreen" in cfg:
        display_dict("Expected downstream screen values", cfg["expectedDownstreamScreen"])
    else:
        display_dict("Configuration", cfg)


## 6. Router and interactive launcher

In [ ]:
def run_simulation(selected_module):
    selected_module = str(selected_module)

    # Validation screens may exist only in frozen packet / external validation JSON.
    if selected_module in {"U", "V", "W", "X", "Y2", "Z", "QG"}:
        cfg = find_screen(selected_module)
        if not cfg:
            # Fall back to SimulationConfigs.json.
            cfg = get_config(selected_module)
        if selected_module == "U":
            return run_module_U(cfg)
        if selected_module == "V":
            return run_module_V(cfg)
        if selected_module == "W":
            return run_module_W(cfg)
        if selected_module == "X":
            return run_module_X(cfg)
        if selected_module == "Y2":
            return run_module_Y2(cfg)
        if selected_module == "Z":
            return run_module_Z(cfg)
        if selected_module == "QG":
            return run_module_QG(cfg)

    cfg = get_config(selected_module)
    mod = cfg.get("module")

    if mod == "G":
        run_module_G(cfg)
    elif mod == "R":
        run_module_R(cfg)
    elif mod == "N":
        run_module_N(cfg)
    elif mod == "S":
        run_module_S(cfg)
    elif mod == "T":
        run_module_T(cfg)
    else:
        run_downstream_kernel_module(cfg)


def canonical_sort_key(name):
    order = {
        "G": 0,
        "R": 1,
        "N": 2,
        "S": 3,
        "T": 4,
        "U": 5,
        "V": 6,
        "W": 7,
        "X": 8,
        "Y2": 9,
        "Z": 10,
        "QG": 11,
        "A": 20,
        "B": 21,
        "C": 22,
        "D": 23,
        "E": 24,
        "F": 25,
        "H": 26,
        "I": 27,
        "J": 28,
        "K": 29,
        "L": 30,
        "M": 31,
        "O": 32,
        "P": 33,
        "Q": 34,
        "G_legacy": 90,
        "N_legacy": 91,
        "R_legacy": 92
    }
    return (order.get(name, 50), name)


config_module_options = [module_key(cfg) for cfg in all_configs]
validation_module_options = [m for m in ["U", "V", "W", "X", "Y2", "Z", "QG"] if find_screen(m)]
module_options = sorted(set(config_module_options + validation_module_options), key=canonical_sort_key)

default_module = "G" if "G" in module_options else module_options[0]

module_selector = Dropdown(
    options=module_options,
    value=default_module,
    description="Module:",
    layout=Layout(width="460px")
)

run_button = Button(description="Run selected module", button_style="primary")
out = Output()


def on_run_clicked(_):
    out.clear_output(wait=True)
    with out:
        run_simulation(module_selector.value)


run_button.on_click(on_run_clicked)

display(VBox([module_selector, run_button, out]))

display(Markdown(
    "Recommended sequence: **G -> R -> N -> S -> T -> U -> V -> W -> X -> Y2 -> Z -> QG**. "
    "Then inspect A-F and H-Q as downstream projections. "
    "Do not use `G_legacy`, `N_legacy`, or `R_legacy` as current derivation modules."
))


## 7. Recommended verification sequence

Use this order when checking the current RFC rebuild:

1. `G` - verify deterministic triadic closure and frozen packet.
2. `R` - verify triad-grouped global closure audit.
3. `N` - verify Module N V2 dimensional projection identities.
4. `S` - verify one-anchor SI bridge.
5. `T` - verify dimensionless coupling map.
6. `U` - verify one-anchor electromagnetic/atomic constant table.
7. `V` - inspect compressed precision-cosmology proxy status.
8. `W` - inspect BBN light-abundance proxy status.
9. `X` - inspect CP/EDM bound and baryon eta proxy status.
10. `Y2` - inspect particle-sector candidate map results. Treat this as exploratory until frozen/retested.
11. `Z` - inspect observer/branching/neural harness. Real EEG/neural data remains required.
12. `QG` - inspect finite spin-foam transition-amplitude audit. This is not a full quantum-gravity proof.

Do not use `G_legacy`, `N_legacy`, or `R_legacy` as current derivation modules.
